In [3]:
import torch

from sc_flow.backends.torch._model import SCFlow
from sc_flow.data.sim import get_dummy_adata
from sc_flow.backends.torch.coupling._coupling import independent_coupling
from sc_flow.backends.torch.probability_paths._probability_paths import LinearDiracProbabilityPath
from sc_flow.data.samplers import FTrainSampler, FValidationSampler

n_obs_pert = 10000
n_obs_ctrl = 5000
adata = get_dummy_adata(n_obs_pert=n_obs_pert, n_obs_ctrl=n_obs_ctrl)

SCFlow.register_adata(
    adata,
    sample_rep="X_tgt",
    conditions={
        "drug": ("drugA", "drugB"),
        "ko": ("koA", "koB"),
    },
    conditions_reps={
        "drug": "drug",
        "ko": "ko",
    },
    # conditions_covariates=["paired_condition"],
    groups=["source_split"],
    groups_encoding={"source_split": "one-hot"},
    control_values_dict={"drug": "control", "ko": "control"},
    source_rep="X_src",
    n_shared_dims=10,
)

model = SCFlow(
    method_id="cfm",
    match_fn=independent_coupling,
    time_sampler=torch.rand,
    probability_path=LinearDiracProbabilityPath(),
    condition_encoder_input_layers={
        "ko": {"output_dim": 16, "hidden_dims": [32, 32]},
        "drug": {"output_dim": 16, "hidden_dims": [32, 32]},
        "source_split": {"output_dim": 16, "hidden_dims": [32, 32]},
    },
)

train_collection = model.dm.compile_adata(adata)
train_sampler = FTrainSampler(train_collection, lambda x: x, n_groups=1, replace_groups=True, replace_samples=True)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()[0]

/Users/lorenzo.consoli/micromamba/envs/sc-flow-tools/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/lorenzo.consoli/micromamba/envs/sc-flow-tools/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 225/225 [00:00<00:00, 3542.86it/s]


In [4]:
model.train()

NotImplementedError: 